# Training Image Classification CNN

This notebook demonstrates:
1. Loading and preprocessing image data
2. Building a CNN from scratch
3. Training with data augmentation
4. Evaluating model performance
5. Visualizing predictions

In [ ]:
# Import libraries
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# Check GPU
print("TensorFlow version:", tf.__version__)
print("GPU Available:", len(tf.config.list_physical_devices('GPU')) > 0)

## 1. Load Image Dataset

In [ ]:
from data_loading.image_dataset import ImageDataset

# Load dataset
dataset = ImageDataset(images_dir='../data/images')
df = dataset.load_data()

print(f"Total images: {len(df)}")
print(f"Classes: {dataset.class_names}")
print(f"Number of classes: {dataset.get_num_classes()}")

In [ ]:
# Split data
train_df, val_df, test_df = dataset.prepare_splits(
    test_size=0.2,
    val_size=0.1,
    random_state=42
)

# Get paths and labels
train_paths, train_labels = dataset.get_paths_and_labels(train_df)
val_paths, val_labels = dataset.get_paths_and_labels(val_df)
test_paths, test_labels = dataset.get_paths_and_labels(test_df)

## 2. Visualize Sample Images

In [ ]:
from preprocessing.image_preprocessing import ImagePreprocessor
from PIL import Image

# Show samples from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for idx, class_name in enumerate(dataset.class_names):
    # Get a sample from this class
    sample_path = train_df[train_df['class_name'] == class_name].sample(1)['file_path'].values[0]
    
    # Load image
    img = Image.open(sample_path)
    
    # Display
    axes[idx].imshow(img)
    axes[idx].set_title(class_name, fontweight='bold')
    axes[idx].axis('off')

# Hide extra subplots
for idx in range(len(dataset.class_names), len(axes)):
    axes[idx].axis('off')

plt.suptitle('Sample Images from Each Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Create Data Pipelines

In [ ]:
from preprocessing.image_preprocessing import create_tf_dataset

# Configuration
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32

# Create preprocessor
preprocessor = ImagePreprocessor(
    target_size=IMAGE_SIZE,
    normalization='standard'
)

# Create TensorFlow datasets
train_dataset = create_tf_dataset(
    train_paths, train_labels, preprocessor,
    batch_size=BATCH_SIZE, shuffle=True
)

val_dataset = create_tf_dataset(
    val_paths, val_labels, preprocessor,
    batch_size=BATCH_SIZE, shuffle=False
)

test_dataset = create_tf_dataset(
    test_paths, test_labels, preprocessor,
    batch_size=BATCH_SIZE, shuffle=False
)

print(f"Training batches: {len(train_dataset)}")
print(f"Validation batches: {len(val_dataset)}")
print(f"Test batches: {len(test_dataset)}")

In [ ]:
# Visualize preprocessed batch
sample_batch = next(iter(train_dataset))
sample_images, sample_labels = sample_batch

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for idx in range(8):
    img = sample_images[idx].numpy()
    label = sample_labels[idx].numpy()
    
    # Denormalize for display
    img_display = (img * 255).astype(np.uint8)
    
    axes[idx].imshow(img_display)
    axes[idx].set_title(f'{dataset.idx_to_class_name(label)}', fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Preprocessed Training Batch', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Build CNN Model

In [ ]:
from models.image_cnn_tf import ImageCNN

# Create model
model = ImageCNN(
    input_shape=(*IMAGE_SIZE, 3),
    num_classes=dataset.get_num_classes(),
    architecture='standard',  # 'simple', 'standard', or 'deep'
    random_state=42
)

# Build model
model.build_model()

## 5. Train the Model

In [ ]:
# Train model
print("Training CNN model...")
print("This may take several minutes depending on your hardware.")

history = model.train(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    epochs=20,
    verbose=1
)

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history.history['accuracy'], label='Train', marker='o', linewidth=2)
ax1.plot(history.history['val_accuracy'], label='Validation', marker='s', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Model Accuracy', fontweight='bold', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(history.history['loss'], label='Train', marker='o', linewidth=2)
ax2.plot(history.history['val_loss'], label='Validation', marker='s', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title('Model Loss', fontweight='bold', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print final metrics
print(f"\nFinal Training Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")

## 6. Evaluate on Test Set

In [ ]:
# Evaluate
test_loss, test_accuracy = model.model.evaluate(test_dataset, verbose=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Get predictions
print("Generating predictions...")
test_preds_probs = model.model.predict(test_dataset, verbose=0)
test_preds = np.argmax(test_preds_probs, axis=1)

# Classification report
print("\nClassification Report:")
print(classification_report(
    test_labels,
    test_preds,
    target_names=dataset.class_names,
    digits=4
))

In [ ]:
# Confusion matrix
cm = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=dataset.class_names,
    yticklabels=dataset.class_names
)
plt.title('Confusion Matrix - Image CNN', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Visualize Predictions

In [ ]:
# Load a few test images for visualization
num_samples = 12
sample_indices = np.random.choice(len(test_paths), num_samples, replace=False)

sample_images = []
for idx in sample_indices:
    img = preprocessor.load_image(test_paths[idx])
    sample_images.append(img)

sample_images = np.array(sample_images)
sample_preds_probs = model.predict_proba(sample_images)
sample_preds = np.argmax(sample_preds_probs, axis=1)
sample_true = test_labels[sample_indices]

# Visualize
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()

for idx in range(num_samples):
    img = sample_images[idx]
    true_label = sample_true[idx]
    pred_label = sample_preds[idx]
    confidence = sample_preds_probs[idx][pred_label]
    
    # Denormalize for display
    img_display = (img * 255).astype(np.uint8)
    
    axes[idx].imshow(img_display)
    
    true_class = dataset.idx_to_class_name(true_label)
    pred_class = dataset.idx_to_class_name(pred_label)
    
    # Color: green if correct, red if wrong
    color = 'green' if true_label == pred_label else 'red'
    
    title = f'True: {true_class}\nPred: {pred_class}\n{confidence*100:.1f}%'
    axes[idx].set_title(title, fontsize=9, color=color, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Model Summary

In [ ]:
print("="*70)
print("Model Summary")
print("="*70)
print(f"Architecture: {model.architecture}")
print(f"Total Parameters: {model.model.count_params():,}")
print(f"Image Size: {IMAGE_SIZE}")
print(f"Number of Classes: {dataset.get_num_classes()}")
print(f"\nPerformance:")
print(f"  Test Accuracy: {test_accuracy:.4f}")
print(f"  Test Loss: {test_loss:.4f}")

# Per-class accuracy
print(f"\nPer-Class Accuracy:")
for i, class_name in enumerate(dataset.class_names):
    class_mask = test_labels == i
    class_acc = (test_preds[class_mask] == test_labels[class_mask]).mean()
    print(f"  {class_name:15s}: {class_acc:.4f}")

## 9. Save Model (Optional)

In [ ]:
# Uncomment to save the model
# model.save('../models/image_cnn_model_notebook.h5')
# print("Model saved!")

## 10. Conclusions

Key takeaways:

1. **CNN Architecture**: Built a custom CNN from scratch with multiple convolutional blocks
2. **Data Pipeline**: Used TensorFlow's efficient data pipeline for fast training
3. **Training**: Applied techniques like batch normalization, dropout, and callbacks
4. **Performance**: Evaluated model with accuracy, confusion matrix, and per-class metrics

Next steps to improve:
- Add data augmentation (rotation, flip, zoom)
- Try different architectures (deeper networks)
- Use transfer learning (VGG16, ResNet, etc.)
- Collect more training data
- Fine-tune hyperparameters (learning rate, batch size, etc.)